In [1]:
%pip install pandas numpy matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.


# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [6]:
import pandas as pd

# Pointing directly to your project folder on the Desktop
file_path = 'Desktop/classwork/dsc-course0-m8-lab/AviationData.csv'

df = pd.read_csv(file_path, encoding='latin-1', low_memory=False)
df.head()

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  str    
 1   Investigation.Type      88889 non-null  str    
 2   Accident.Number         88889 non-null  str    
 3   Event.Date              88889 non-null  str    
 4   Location                88837 non-null  str    
 5   Country                 88663 non-null  str    
 6   Latitude                34382 non-null  str    
 7   Longitude               34373 non-null  str    
 8   Airport.Code            50132 non-null  str    
 9   Airport.Name            52704 non-null  str    
 10  Injury.Severity         87889 non-null  str    
 11  Aircraft.damage         85695 non-null  str    
 12  Aircraft.Category       32287 non-null  str    
 13  Registration.Number     87507 non-null  str    
 14  Make                    88826 non-null  str    
 

In [10]:
#Checking the different types of aircraft categories available
df["Aircraft.Category"].value_counts(dropna = False)

Aircraft.Category
NaN                  56602
Airplane             27617
Helicopter            3440
Glider                 508
Balloon                231
Gyrocraft              173
Weight-Shift           161
Powered Parachute       91
Ultralight              30
Unknown                 14
WSFT                     9
Powered-Lift             5
Blimp                    4
UNK                      2
Rocket                   1
ULTR                     1
Name: count, dtype: int64

In [14]:
#Filtering the dataset to keep only explicit airplanes or cases where it is blank  but has an engine count of a standard airplane
airplane_df = df[(df["Aircraft.Category"] == "Airplane")  |
    ((df["Aircraft.Category"].isna()) & (df["Number.of.Engines"].isin([1.0,2.0,3.0,4.0])))]
#Checking how many rows we have left after filtering
print("Filtered dataset rows:", len(airplane_df))
airplane_df.head()
# we have 80,873 rows and 31 columns in our dataset

Filtered dataset rows: 80873


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
5,20170710X52551,Accident,NYC79AA106,1979-09-17,"BOSTON, MA",United States,42.445277,-70.758333,NaN,NaN,...,NaN,Air Canada,NaN,NaN,1.0,44.0,VMC,Climb,Probable Cause,19-09-2017


# If a record is blank but the number of engines column says 1,2,3 or 4 its highly a standard plane

In [15]:
# Checking for missing values in the injury column within the filtered airplane data
airplane_df[["Total.Fatal.Injuries" , "Total.Serious.Injuries", "Total.Minor.Injuries", "Total.Uninjured"]].isna().sum()

Total.Fatal.Injuries      10138
Total.Serious.Injuries    11071
Total.Minor.Injuries      10447
Total.Uninjured            4988
dtype: int64

In [20]:
# 1. Filling the missing injury/uninjured columns with 0 so that the math doesn't break
injury_cols = ["Total.Fatal.Injuries", "Total.Serious.Injuries", "Total.Minor.Injuries","Total.Uninjured"]
airplane_df[injury_cols] = airplane_df[injury_cols].fillna(0)
# 2. Calculating the total estimated people on board each flight
airplane_df["Total.Passengers"] = airplane_df[injury_cols].sum(axis = 1)
# 3. Creating a metric for the severe injury rate(fatal + serious) per flight
#    Add a small check just to make sure we don't divide by zero if a row has empty data
airplane_df["Severe.Injury.Rate"] = (airplane_df["Total.Fatal.Injuries"] + airplane_df["Total.Serious.Injuries"]) / airplane_df["Total.Passengers"]
airplane_df["Severe.Injury.Rate"] = airplane_df["Severe.Injury.Rate"].fillna(0)
#    Checking the newley created columns
airplane_df[["Total.Passengers","Severe.Injury.Rate"]].head()

,Total.Passengers,Severe.Injury.Rate
0,2.0,1.0
1,4.0,1.0
2,3.0,1.0
3,2.0,1.0
5,45.0,0.0


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [25]:
# 1. Filling missing value in individual injury colums with 0
injury_cols = ["Total.Fatal.Injuries", "Total.Serious.Injuries", "Total.Minor.Injuries", "Total.Uninjured"]
airplane_df[injury_cols] = airplane_df[injury_cols].fillna(0)
# 2. Calculating the total estimated people on board each flight
airplane_df["Total.Passengers"] = airplane_df[injury_cols].sum(axis =1)
# 3. Creating the fractional severe injury rate metric (Fatal+serious)
#    used np. to cleanly handle rows where Total.Passengers is 0 without creating NANs or infinities
import numpy as np
airplane_df["Severe.Injury.Rate"] = np.where (
    airplane_df["Total.Passengers"] > 0,
     (airplane_df["Total.Fatal.Injuries"] + airplane_df["Total.Serious.Injuries"])  /  airplane_df["Total.Passengers"],
      0.0
     )      
# 4. Filtering out any unrealistic rows if they exist (e.g., negative passenger counts)
airplane_df = airplane_df[airplane_df["Total.Passengers"] >= 0]
#    Preview to verify if it is clean
airplane_df[["Total.Passengers", "Severe.Injury.Rate"]].head()

,Total.Passengers,Severe.Injury.Rate
0,2.0,1.0
1,4.0,1.0
2,3.0,1.0
3,2.0,1.0
5,45.0,0.0


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [ ]:
# 1. Inspect the value distribution and find any missing data
# 2. Build a binary tracking column (is_destroyed) where 1 indicates Aircraft destroyed and 0 indicates it wasn't

In [32]:
# 1. Inspect the unique values and look for missing entries
print("--- Value Counts Before Cleaning ---")
print(airplane_df["Aircraft.damage"].value_counts(dropna = False))
# 2. Imputing missing damaged entries woth "Unknown" so string parsing it doesn't give errors
airplane_df["Aircraft.damage"] = airplane_df["Aircraft.damage"].fillna("Unknown")
# 3 Creating a derived binary column:1 if the plane was destroyed , 0 otherwise
#   Using .str.strip().lower() prevents spacing or casing mismatch
airplane_df["Is_Destroyed"] = airplane_df["Aircraft.damage"].apply(
    lambda x: 1 if str(x).strip().lower() == "destroyed" else 0
)
# 4 Preview the changes made
print("n\---Cleaned Feature Preview ---")
airplane_df[["Aircraft.damage" ,"Is_Destroyed"]].head(10)

--- Value Counts Before Cleaning ---
Aircraft.damage
Substantial    58929
Destroyed      17017
Minor           2420
NaN             2407
Unknown          100
Name: count, dtype: int64
n\---Cleaned Feature Preview ---


<>:12: SyntaxWarning: "\-" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\-"? A raw string is also an option.
<>:12: SyntaxWarning: "\-" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\-"? A raw string is also an option.
C:\Users\Admin\AppData\Local\Temp\ipykernel_20132\2645965368.py:12: SyntaxWarning: "\-" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\-"? A raw string is also an option.
  print("n\---Cleaned Feature Preview ---")


,Aircraft.damage,Is_Destroyed
0,Destroyed,1
1,Destroyed,1
2,Destroyed,1
3,Destroyed,1
5,Substantial,0
6,Destroyed,1
7,Substantial,0
8,Substantial,0
9,Substantial,0
10,Substantial,0


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [33]:
# 1. Cleaning missing values and standardize capitalization / whitespace
airplane_df["Make"] = airplane_df["Make"].fillna("Unknown").astype(str).str.strip().str.title()

# 2. Inspecting the total count of unique manufacturers before filtering
print(f"Total unique manufacturers initially: {airplane_df['Make'].nunique()}")

# 3. Calculating frequency counts for each manufacturer
make_counts = airplane_df["Make"].value_counts()

# 4. Filter data to only keep Makes with 50 or more incidents
frequent_makes = make_counts[make_counts >= 50].index
airplane_df = airplane_df[airplane_df["Make"].isin(frequent_makes)]

# 5. Printing a summary check for the client
print(f"Total unique manufacturers after threshold filtering (>= 50): {airplane_df['Make'].nunique()}")
print("\n--- Top 10 Manufacturers remaining in dataset ---")
print(airplane_df["Make"].value_counts().head(10))

Total unique manufacturers initially: 6892
Total unique manufacturers after threshold filtering (>= 50): 85

--- Top 10 Manufacturers remaining in dataset ---
Make
Cessna         26724
Piper          14625
Beech           5257
Boeing          2401
Bell            1675
Mooney          1323
Grumman         1166
Bellanca        1037
Air Tractor      680
Hughes           662
Name: count, dtype: int64


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [34]:
# 1. Filling missing Model values with 'Unknown' and clean up white space/capitalization
airplane_df["Model"] = airplane_df["Model"].fillna("Unknown").astype(str).str.strip().str.upper()

# 2. Checking how many unique model designations exist now
print(f"Total unique raw model labels: {airplane_df['Model'].nunique()}")

# 3. Creating a unique identifier combining Make and Model to ensure labels are unique to each manufacturer
airplane_df["Aircraft_Type"] = airplane_df["Make"] + "-" + airplane_df["Model"]

# 4. Preview the resulting mapping
print("\n--- Unique Aircraft Type Identification Preview ---")
airplane_df[["Make", "Model", "Aircraft_Type"]].head(10)

Total unique raw model labels: 5455

--- Unique Aircraft Type Identification Preview ---


,Make,Model,Aircraft_Type
0,Stinson,108-3,Stinson-108-3
1,Piper,PA24-180,Piper-PA24-180
2,Cessna,172M,Cessna-172M
3,Rockwell,112,Rockwell-112
5,Mcdonnell Douglas,DC9,Mcdonnell Douglas-DC9
6,Cessna,180,Cessna-180
7,Cessna,140,Cessna-140
8,Cessna,401B,Cessna-401B
9,North American,NAVION L-17B,North American-NAVION L-17B
10,Piper,PA-28-161,Piper-PA-28-161


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [35]:
# 1. Define categorical columns list
text_cols = ["Engine.Type", "Weather.Condition", "Purpose.of.flight", "Broad.phase.of.flight"]

# 2. Converting all text columns to clean uppercase strings
for col in text_cols:
    if col in airplane_df.columns:
        airplane_df[col] = airplane_df[col].astype(str).str.strip().str.upper()
        # Replacing string 'NAN' or 'UNKNOWN' representations back to true missing values
        airplane_df[col] = airplane_df[col].replace(['NAN', 'UNKNOWN'], np.nan)

# 3. Handling Number.of.Engines missing records reasonably (defaulting to 1.0)
if "Number.of.Engines" in airplane_df.columns:
    airplane_df["Number.of.Engines"] = airplane_df["Number.of.Engines"].fillna(1.0)

# 4. Preview the resulting distributions to confirm standardization
print("--- Standardized Value Distribution Check ---")
print(airplane_df["Weather.Condition"].value_counts(dropna=False).head())
print("\n--- Phase of Flight Distribution Check ---")
print(airplane_df["Broad.phase.of.flight"].value_counts(dropna=False).head())

--- Standardized Value Distribution Check ---
Weather.Condition
VMC    59753
IMC     5332
NaN     2673
UNK      849
Name: count, dtype: int64

--- Phase of Flight Distribution Check ---
Broad.phase.of.flight
NaN            17123
LANDING        13462
TAKEOFF        10272
CRUISE          8807
MANEUVERING     6386
Name: count, dtype: int64


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [36]:
# 1. Defining the columns target for dropping
cols_to_drop = ["Schedule", "Air.carrier", "Publication.Date"]

# 2. Safely dropping them only if they are present in the dataframe to avoid errors
existing_drops = [col for col in cols_to_drop if col in airplane_df.columns]
airplane_df = airplane_df.drop(columns=existing_drops)

# 3. Verify they are gone by looking at the remaining columns list
print("--- Remaining Columns in Dataset ---")
print(airplane_df.columns.tolist())

--- Remaining Columns in Dataset ---
['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date', 'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code', 'Airport.Name', 'Injury.Severity', 'Aircraft.damage', 'Aircraft.Category', 'Registration.Number', 'Make', 'Model', 'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description', 'Purpose.of.flight', 'Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured', 'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status', 'Total.Passengers', 'Severe.Injury.Rate', 'Is_Destroyed', 'Aircraft_Type']


In [38]:
# Checking non-null counts across the entire dataframe to verify the 20,000 rule
non_null_counts = airplane_df.notnull().sum()
columns_violating_rule = non_null_counts[non_null_counts <= 20000]

print("--- Columns with 20,000 or fewer non-nulls ---")
if len(columns_violating_rule) == 0:
    print("Perfect! Every remaining column has more than 20,000 non-null records.")
else:
    print(columns_violating_rule)

--- Columns with 20,000 or fewer non-nulls ---
Perfect! Every remaining column has more than 20,000 non-null records.


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [40]:
# 1. Defining the name for my already cleaned dataset output
output_filename = "cleaned_aviation_accidents.csv"

# 2. Exporting the dataframe to a CSV file without including the row index integers
airplane_df.to_csv(output_filename, index=False)

# 3. Verification confirmation finally!
print(f"Success! Your clean dataset has been successfully saved as: '{output_filename}'")
print(f"Final shape of the processed data: {airplane_df.shape}")

Success! Your clean dataset has been successfully saved as: 'cleaned_aviation_accidents.csv'
Final shape of the processed data: (68607, 32)
